In [0]:
%run ./connectionNotebook

In [0]:
from pyspark.sql.functions import col, current_timestamp, row_number
from pyspark.sql.window import Window

bronze_path = "abfss://bronze@adlsg2rag.dfs.core.windows.net/sqlserver/products/load_date=2026-03-13/"

In [0]:
df = spark.read.format("parquet").load(bronze_path)
df.printSchema()

In [0]:
# PK validation
df_clean = df.filter(col("ProductID").isNotNull() & (col("ProductID") != 0))

In [0]:
# Processing timestamp
df_clean = df_clean.withColumn("processed_ts", current_timestamp())

In [0]:
# Duplicate check
w = Window.partitionBy("ProductID").orderBy(col("processed_ts").desc())
df_clean = (
    df_clean.withColumn("rn", row_number().over(w))
            .filter(col("rn") == 1)
            .drop("rn")
)

In [0]:
# Select and rename columns
df_clean = df_clean.select(
    col("ProductID").alias("src_ProductID"),
    col("ProductName").alias("src_ProductName"),
    col("Category").alias("src_Category"),
    col("Price").alias("src_Price"),
    col("CreatedDate").alias("src_CreatedDate"),
    col("processed_ts")
)

In [0]:
catalog_name = 'adbrag'
schema_name = 'silver'

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.products_silver
(
    src_ProductID INT,
    src_ProductName STRING,
    src_Category STRING,
    src_Price DECIMAL(10,2),
    src_CreatedDate TIMESTAMP,
    processed_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_ProductID)
""")



In [0]:
df_clean.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{schema_name}.products_silver"
)